In [1]:
!python --version

Python 3.11.5


NHTSA is the official source for the “what goes wrong with this car” part of Mile Zero. Matching a vibe to a car is only half the job. The other half is helping someone buy informed, and that needs real information about known problems.

Why NHTSA works for that:

It’s official and trustworthy. It’s the U.S. government’s vehicle safety agency. Owners file complaints there, and each complaint is tagged with the part that failed (engine, brakes, electrical, and so on).
It’s free and needs no key. Anyone can rerun your code, which the course’s reproducibility requirement asks for.
It avoids scraping. We said earlier we’d stay away from forum scraping because of terms-of-service and fragility problems. This is a public API meant to be used.

In [2]:
#Installing packages for working with data
import requests, pandas as pd
print(requests.__version__, pd.__version__)

2.34.2 3.0.6


In [3]:
url = "https://api.nhtsa.gov/complaints/complaintsByVehicle"
params = {"make": "ACURA", "model": "RSX", "modelYear": 2002}

response = requests.get(url, params=params, timeout=30)
print(response.status_code)

data = response.json() #Converts the response to a Python dictionary
print(list(data.keys()))

200
['count', 'message', 'results']


In [4]:
print(data["count"])
print(data["message"])
first = data["results"][0]
print(list(first.keys()))
#93 Users filed cmoplaints about 2002 Acura RSX vehicles. 

93
Results returned successfully
['odiNumber', 'manufacturer', 'crash', 'fire', 'numberOfInjuries', 'numberOfDeaths', 'dateOfIncident', 'dateComplaintFiled', 'vin', 'components', 'summary', 'products']


In [5]:
#Odometer fraud, but labelled as electrical system. 
print(first["components"])
print(first["crash"], first["fire"])
print(first["summary"])

ELECTRICAL SYSTEM
False False
Odometer Fraud. The contact owned a 2002 Acura SRX. The contact stated that the vehicle was sold to a friend in a private sale. The new owner then attempted to sell the vehicle to another person, and upon obtaining the Carfax report, it was discovered that there was a mileage discrepancy. At the time of purchase, the vehicle mileage was approximately 115,000, and at the time of registration, it was discovered that the mileage was 213,288.


In [6]:
for c in data["results"][:10]:
    print(c["components"], "|", c["summary"][:120])
    print()

ELECTRICAL SYSTEM | Odometer Fraud. The contact owned a 2002 Acura SRX. The contact stated that the vehicle was sold to a friend in a privat

STEERING,ELECTRICAL SYSTEM | Duralast 12volt battery post are smaller than they should be. It cause the battery terminal to came loose after a hard b

SEATS | The contact owns a 2022 Acura RSX. The contact stated that while the vehicle was parked, the contact became aware that t

POWER TRAIN | The car stopped at an traffic intersection when making a left turn. Fortunately, the traffic was light. Later, a broken 

AIR BAGS,FORWARD COLLISION AVOIDANCE | THE LIGHT FOR MY AIR BAG IS ON AND THE PASSENGER AIR BAG LIGHT FLICKERS ON TOO.  THE CRUSE CONTROL DOES NOT WORK.

FUEL/PROPULSION SYSTEM | MY CAR HAD A STRONG GAS SMELL INSIDE AND OUTSIDE VEHICLE WHENEVER I DROVE IT. I LOOKED AND FOUND THAT THE GAS PUMP PLAST

FUEL/PROPULSION SYSTEM | WHILE DRIVING I STARTED TO SMELL FUEL IN THE PASSENGER COMPARTMENT. UPON INSPECTION OF THE FUEL PUMP, I DISCOVERED 

In [7]:
df = pd.DataFrame(data["results"])
print(df.shape)
print(df["components"].value_counts().head(10))

(93, 12)
components
POWER TRAIN                  24
ENGINE AND ENGINE COOLING    11
AIR BAGS                      9
STRUCTURE                     4
EXTERIOR LIGHTING             3
SERVICE BRAKES, HYDRAULIC     3
SEATS                         2
FUEL/PROPULSION SYSTEM        2
STEERING                      2
AIR BAGS,SEAT BELTS           2
Name: count, dtype: int64


In [8]:
import re

df["component_list"] = df["components"].apply(lambda s: re.split(r",(?! )", s))
print(df["component_list"].explode().value_counts().head(10))

component_list
POWER TRAIN                  30
ENGINE AND ENGINE COOLING    20
AIR BAGS                     14
STRUCTURE                     6
ELECTRICAL SYSTEM             4
EXTERIOR LIGHTING             4
SERVICE BRAKES, HYDRAULIC     4
STEERING                      3
SEATS                         3
FUEL/PROPULSION SYSTEM        3
Name: count, dtype: int64


In [9]:
power = df[df["components"].str.contains("POWER TRAIN")]
print(len(power))

for s in power["summary"].head(8):
    print(s[:200])
    print()

30
The car stopped at an traffic intersection when making a left turn. Fortunately, the traffic was light. Later, a broken driver side cv axel was identified, which had sever corrosion inside the damper.

TL* THE CONTACT OWNS A 2002 ACURA RSX TYPE S. THE CONTACT STATED THAT WHILE DRIVING APPROXIMATELY 10-60 MPH, THE THIRD GEAR SHIFTED OUT OF GEAR. IN ADDITION, THERE WAS A VERY LOUD GRINDING NOISE. THE 

2002 FORD EXPLORER XLT 4.0 LITER JERKS WHEN ENGAGING INTO REVERSE. VEHICLE JERKS HARD ENOUGH THAT THE BRAKES ARE NOT ENOUGH TO STOP IT FROM MOVING. NO OTHER GEAR IS LIKE THIS. UPON FURTHER STUDY FROM 

TL*THE CONTACT OWNS A 2002 ACURA RSX.  WHEN SWITCHING FROM SECOND TO THIRD GEAR, THE SHIFTER WOULD POP OUT OF GEAR AND MAKE A GRINDING NOISE.  THE MANUFACTURER COULD NOT ASSIST BECAUSE THERE WERE NO R

I'VE HAD TRANSMISSION PROBLEMS EVER SINCE I PURCHASED THE NEW RSX IN 02... THIS PROBLEM HAS BEEN WIDELY REPORTED ON THE INTERNET... HOWEVER ACURA HAS FAILED TO ADDRESS THIS ISSUE... THE PRO

vin is the independent identifier. A VIN is the vehicle identification number stamped on every car. These are cut to 11 characters, but that still carries a lot:
The first 3 characters identify the manufacturer. JH4 is Acura.
The 10th character is the model year. In JH4DC53052C it’s 2, which means 2002.
Characters 4 to 8 describe the model. I think DC5 is the RSX’s chassis code, but that’s from memory, so we’ll test it against the data.

In [10]:
df["vin_prefix"] = df["vin"].str.upper().str[:6]
print(df["vin_prefix"].value_counts(dropna=False).head(10))
print()

wrong = df[df["summary"].str.contains("EXPLORER|MDX|SRX", case=False, na=False)]
for _, row in wrong.iterrows():
    print(row["vin"], "|", row["summary"][:60])

vin_prefix
JH4DC5    67
          20
2HNYD1     3
PLEASE     1
JH4DD5     1
19UUA5     1
Name: count, dtype: int64

JH4DC53052C | Odometer Fraud. The contact owned a 2002 Acura SRX. The cont
 | 2002 FORD EXPLORER XLT 4.0 LITER JERKS WHEN ENGAGING INTO RE
 | MY 2002 MDX HAS 75,000 MILES AND NEEDS THE ENGINE MOUNTS REP
2HNYD18852H | MULTIPLE ENGINE MOUNT FAILURES ON 2002 ACURA MDX. FIRST FAIL
2HNYD18402H | THE 2001 ACURA MDX STALLS AT HIGH SPEEDS AND THEN KICKS BACK


In [11]:
vin = df["vin"].fillna("").str.strip().str.upper()
vin_ok = vin.str.startswith("JH4DC5")
text_ok = df["summary"].str.contains("RSX", case=False, na=False)

keep = vin_ok | ((vin == "") & text_ok)
print(keep.sum(), "kept of", len(df))
print()

for _, row in df[~keep].head(12).iterrows():
    print(row["vin"], "|", row["summary"][:60])

73 kept of 93

 | 2002 FORD EXPLORER XLT 4.0 LITER JERKS WHEN ENGAGING INTO RE
 | MY BUSINESS PURCHASED TWO BRAND NEW 2002 ACURA VEHICLES, A C
 | MY 2002 MDX HAS 75,000 MILES AND NEEDS THE ENGINE MOUNTS REP
2HNYD18892H | MIL COMES ON WITH DTC P1457 - SIGNALING EVAP BYPASS SOLENOID
2HNYD18852H | MULTIPLE ENGINE MOUNT FAILURES ON 2002 ACURA MDX. FIRST FAIL
2HNYD18402H | THE 2001 ACURA MDX STALLS AT HIGH SPEEDS AND THEN KICKS BACK
 | I PURCHASED 4 SIGMA REGENT TOURING 205/55TR16, 7E55 FROM TIR
 | DT: CONTACT WAS DRIVING ON THE FREEWAY AND HE COULDN'T STEER
 | OEM FOGLIGHT LENSES, CRACKS TOO EASILY.  *AK
 | SECOND GEAR GRINDING LED TO TRANSMISSION FAILURE.  NOW  I HA
 | IN THE 12 MONTHS I HAVE HAD MY CAR, IT HAS BEEN IN THE SHOP 
 | WHILE DRIVING  OVER BUMPS AND POTHOLES ABS LIGHT WOULD COME 


In [12]:
df["match_confidence"] = keep.map({True: "confident", False: "unverified"})
print(df["match_confidence"].value_counts())

match_confidence
confident     73
unverified    20
Name: count, dtype: int64


In [13]:
frames = []
for year in range(2002, 2007):
    r = requests.get(url, params={"make": "ACURA", "model": "RSX", "modelYear": year}, timeout=30)
    year_df = pd.DataFrame(r.json()["results"])
    year_df["model_year"] = year
    frames.append(year_df)
    print(year, len(year_df))

rsx = pd.concat(frames, ignore_index=True)
print("total:", len(rsx))

2002 93
2003 31
2004 17
2005 32
2006 24
total: 197


In [14]:
#Sucker Car Comparison
def get_complaints(make, model, years):
    frames = []
    for year in years:
        r = requests.get(url, params={"make": make, "model": model, "modelYear": year}, timeout=30)
        d = pd.DataFrame(r.json()["results"])
        d["model_year"] = year
        frames.append(d)
        print(model, year, len(d))
    return pd.concat(frames, ignore_index=True)

eclipse = get_complaints("MITSUBISHI", "ECLIPSE", range(2002, 2006))
print("total:", len(eclipse))

ECLIPSE 2002 104
ECLIPSE 2003 166
ECLIPSE 2004 14
ECLIPSE 2005 6
total: 290


In [15]:
def component_share(d, label):
    comps = d["components"].fillna("").apply(lambda s: re.split(r",(?! )", s)).explode()
    comps = comps[comps != ""]
    return (comps.value_counts() / len(d) * 100).round(1).rename(label)

rsx_0205 = rsx[rsx["model_year"] <= 2005]
compare = pd.concat(
    [component_share(rsx_0205, "RSX %"), component_share(eclipse, "Eclipse %")], axis=1
).fillna(0)
print(compare.sort_values("RSX %", ascending=False).head(10))

                           RSX %  Eclipse %
components                                 
POWER TRAIN                 26.6       10.7
AIR BAGS                    18.5        8.3
ENGINE AND ENGINE COOLING   15.0        6.9
SUSPENSION                   9.2        1.4
TIRES                        6.4        8.3
ELECTRICAL SYSTEM            4.6        6.2
SERVICE BRAKES, HYDRAULIC    4.6        3.1
STEERING                     4.0        3.1
EXTERIOR LIGHTING            4.0        3.1
UNKNOWN OR OTHER             3.5        5.2


In [16]:
e = eclipse.assign(c=eclipse["components"].fillna("").apply(lambda s: re.split(r",(?! )", s))).explode("c")
print(e[e["model_year"] == 2003]["c"].value_counts().head(5))
print()
print(e[e["model_year"] != 2003]["c"].value_counts().head(5))

c
STRUCTURE                    40
POWER TRAIN                  25
AIR BAGS                     15
TIRES                        15
ENGINE AND ENGINE COOLING    14
Name: count, dtype: int64

c
WHEELS               39
STRUCTURE            26
ELECTRICAL SYSTEM     9
AIR BAGS              9
TIRES                 9
Name: count, dtype: int64


In [17]:
#rsx.info()
#print()
print(rsx.iloc[1])

odiNumber                                                      11751583
manufacturer                           Honda (American Honda Motor Co.)
crash                                                             False
fire                                                              False
numberOfInjuries                                                      0
numberOfDeaths                                                        0
dateOfIncident                                               07/19/2026
dateComplaintFiled                                           07/20/2026
vin                                                         jh4dc53802c
components                                   STEERING,ELECTRICAL SYSTEM
summary               Duralast 12volt battery post are smaller than ...
products              [{'type': 'Vehicle', 'productYear': '2002', 'p...
model_year                                                         2002
Name: 1, dtype: object


In [18]:
print(rsx["products"].apply(len).value_counts())
print()
print(rsx["products"].explode().apply(lambda p: p["type"]).value_counts())

products
1    187
2     10
Name: count, dtype: int64

products
Vehicle      197
Equipment      5
Tire           5
Name: count, dtype: int64


In [19]:
def top_components(make, model, years, n=5):
    d = get_complaints(make, model, years)
    return len(d), component_share(d, "share %").head(n)

total, top = top_components("ACURA", "RSX", range(2002, 2007))
print(total)
print(top)

RSX 2002 93
RSX 2003 31
RSX 2004 17
RSX 2005 32
RSX 2006 24
197
components
POWER TRAIN                  25.9
AIR BAGS                     17.3
ENGINE AND ENGINE COOLING    13.7
SUSPENSION                   10.7
EXTERIOR LIGHTING             6.1
Name: share %, dtype: float64


In [20]:
def mech_share(d):
    hit = d["components"].fillna("").str.contains("POWER TRAIN|ENGINE AND ENGINE COOLING")
    return round(hit.mean() * 100, 1)

print(mech_share(rsx))

38.1


In [21]:
#Identify popular cars from fast and furious movies (2000s)
cars = pd.DataFrame([
    ("eclipse_2g",  "MITSUBISHI", "ECLIPSE",  1995, 1999, True,  "2001"),
    ("eclipse_3g",  "MITSUBISHI", "ECLIPSE",  2000, 2005, True,  "2003"),
    ("rx7_fd",      "MAZDA",      "RX-7",     1993, 1995, True,  "2001, 2003, 2006"),
    ("nissan_240sx","NISSAN",     "240SX",    1995, 1998, True,  "2001"),
    ("integra",     "ACURA",      "INTEGRA",  1994, 2001, True,  "2001, 2003"),
    ("supra_a80",   "TOYOTA",     "SUPRA",    1993, 1998, True,  "2001, 2003"),
    ("civic_eg",    "HONDA",      "CIVIC",    1992, 1995, True,  "2001, 2003"),
    ("s2000",       "HONDA",      "S2000",    2000, 2006, True,  "2003"),
    ("nsx",         "ACURA",      "NSX",      1991, 2005, True,  "2003"),
    ("nissan_350z", "NISSAN",     "350Z",     2003, 2006, True,  "2006"),
    ("mazda_rx8",   "MAZDA",      "RX-8",     2004, 2006, True,  "2006"),
    ("evo_8_9",     "MITSUBISHI", "LANCER",   2003, 2006, True,  "2006"),
    ("skyline_gtr", "NISSAN",     "SKYLINE GT-R", 1995, 2002, False, "2003"),
    ("evo_7",       "MITSUBISHI", "LANCER EVOLUTION VII", 2001, 2003, False, "2003"),
    ("silvia_s15",  "NISSAN",     "SILVIA S15", 1999, 2002, False, "2006"),
], columns=["car_id", "make", "model", "year_start", "year_end", "in_nhtsa", "films"])

print(cars)

          car_id        make                 model  year_start  year_end  \
0     eclipse_2g  MITSUBISHI               ECLIPSE        1995      1999   
1     eclipse_3g  MITSUBISHI               ECLIPSE        2000      2005   
2         rx7_fd       MAZDA                  RX-7        1993      1995   
3   nissan_240sx      NISSAN                 240SX        1995      1998   
4        integra       ACURA               INTEGRA        1994      2001   
5      supra_a80      TOYOTA                 SUPRA        1993      1998   
6       civic_eg       HONDA                 CIVIC        1992      1995   
7          s2000       HONDA                 S2000        2000      2006   
8            nsx       ACURA                   NSX        1991      2005   
9    nissan_350z      NISSAN                  350Z        2003      2006   
10     mazda_rx8       MAZDA                  RX-8        2004      2006   
11       evo_8_9  MITSUBISHI                LANCER        2003      2006   
12   skyline

In [22]:
import io, time
from contextlib import redirect_stdout

rows = []
for car in cars[cars["in_nhtsa"]].itertuples():
    with redirect_stdout(io.StringIO()):   # hides the per-year prints
        d = get_complaints(car.make, car.model, range(car.year_start, car.year_end + 1))
    if len(d) == 0 or "components" not in d:
        rows.append((car.car_id, 0, None, None))
    else:
        top = component_share(d, "share").index[0]
        rows.append((car.car_id, len(d), mech_share(d), top))
    print(car.car_id, rows[-1][1])
    time.sleep(0.3)

results = pd.DataFrame(rows, columns=["car_id", "complaints", "mech_share_%", "top_component"])
print(results)
#Some cars do not have enough data as supra has only 16 complaints and NSX only 9. 

eclipse_2g 1050
eclipse_3g 920
rx7_fd 121
nissan_240sx 105
integra 260
supra_a80 16
civic_eg 752
s2000 156
nsx 9
nissan_350z 555
mazda_rx8 743
evo_8_9 452
          car_id  complaints  mech_share_%              top_component
0     eclipse_2g        1050          32.2  ENGINE AND ENGINE COOLING
1     eclipse_3g         920          13.8                     WHEELS
2         rx7_fd         121          43.0  ENGINE AND ENGINE COOLING
3   nissan_240sx         105           8.6                   AIR BAGS
4        integra         260          13.5                   AIR BAGS
5      supra_a80          16          18.8                      TIRES
6       civic_eg         752          11.7                   AIR BAGS
7          s2000         156          28.8                POWER TRAIN
8            nsx           9          33.3  SERVICE BRAKES, HYDRAULIC
9    nissan_350z         555          23.4                 SUSPENSION
10     mazda_rx8         743          49.0                POWER TRAIN
11   

In [23]:
import os
os.makedirs("data", exist_ok=True)

summary = cars.merge(results, on="car_id", how="left")
summary.to_csv("data/car_summary.csv", index=False)
print(summary[["car_id", "in_nhtsa", "complaints", "mech_share_%"]])

          car_id  in_nhtsa  complaints  mech_share_%
0     eclipse_2g      True      1050.0          32.2
1     eclipse_3g      True       920.0          13.8
2         rx7_fd      True       121.0          43.0
3   nissan_240sx      True       105.0           8.6
4        integra      True       260.0          13.5
5      supra_a80      True        16.0          18.8
6       civic_eg      True       752.0          11.7
7          s2000      True       156.0          28.8
8            nsx      True         9.0          33.3
9    nissan_350z      True       555.0          23.4
10     mazda_rx8      True       743.0          49.0
11       evo_8_9      True       452.0          33.8
12   skyline_gtr     False         NaN           NaN
13         evo_7     False         NaN           NaN
14    silvia_s15     False         NaN           NaN


In [24]:
#Image Database via wikipedia
api = "https://commons.wikimedia.org/w/api.php"
headers = {"User-Agent": "MileZero-class-project/0.1 (DATA 785; github.com/ManKNg)"}
params = {
    "action": "query", "format": "json",
    "generator": "search", "gsrsearch": "Mazda RX-7 FD", "gsrnamespace": 6, "gsrlimit": 10,
    "prop": "imageinfo", "iiprop": "url|size|extmetadata",
}

r = requests.get(api, params=params, headers=headers, timeout=30)
print(r.status_code)

for p in r.json()["query"]["pages"].values():
    info = p["imageinfo"][0]
    lic = info["extmetadata"].get("LicenseShortName", {}).get("value")
    print(p["title"], "|", lic, "|", info["width"], "x", info["height"])

200
File:2000-2003 Mazda RX-7 engine.jpg | CC0 | 5672 x 3781
File:2017-06-10 Jason Clark RX-7.JPG | Attribution | 2265 x 1500
File:FD Mazda RX-7.jpg | CC BY 2.0 | 4288 x 2848
File:FD Mazda RX-7 rear.jpg | CC BY 2.0 | 5184 x 3456
File:Mazda RX-7 (FD) Washington DC Metro Area, USA (1).jpg | CC BY 4.0 | 2700 x 2160
File:Mazda RX-7 (FD) Washington DC Metro Area, USA (1.1).jpg | CC BY 4.0 | 2155 x 1724
File:Mazda RX-7 (FD) at the 2025 Shannons Adelaide Rally (028A4672).jpg | CC BY 4.0 | 6960 x 4640
File:Mazda RX-7 (FD) at the Pavilion Hotel Kuala Lumpur, Malaysia (DSCF1038).jpg | CC BY-SA 4.0 | 7599 x 5066
File:Mazda RX-7 (FD) at the Pavilion Hotel Kuala Lumpur, Malaysia (DSCF1048).jpg | CC BY-SA 4.0 | 3888 x 2592
File:Mazda RX-7 of Mike Whiddett at the 2026 Adelaide Motorsport Festival (DSCF2622).jpg | CC BY-SA 4.0 | 7728 x 5152


In [25]:
params["iiurlwidth"] = 640
r = requests.get(api, params=params, headers=headers, timeout=30)

for p in r.json()["query"]["pages"].values():
    info = p["imageinfo"][0]
    meta = info["extmetadata"]
    print(p["title"])
    print("  artist:", meta.get("Artist", {}).get("value", "")[:80])
    print("  license:", meta.get("LicenseShortName", {}).get("value"), "|", meta.get("LicenseUrl", {}).get("value"))
    print("  attribution required:", meta.get("AttributionRequired", {}).get("value"))
    print("  thumb:", info.get("thumburl"))

File:2000-2003 Mazda RX-7 engine.jpg
  artist: <a href="//commons.wikimedia.org/wiki/User:TTTNIS" title="User:TTTNIS">TTTNIS</a
  license: CC0 | http://creativecommons.org/publicdomain/zero/1.0/deed.en
  attribution required: false
  thumb: https://thumb.wikimedia.org/wikipedia/commons/thumb/c/cb/2000-2003_Mazda_RX-7_engine.jpg/960px-2000-2003_Mazda_RX-7_engine.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail
File:2017-06-10 Jason Clark RX-7.JPG
  artist: <a href="//commons.wikimedia.org/wiki/User:Collard" title="User:Collard">Lewis C
  license: Attribution | None
  attribution required: None
  thumb: https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8f/2017-06-10_Jason_Clark_RX-7.JPG/960px-2017-06-10_Jason_Clark_RX-7.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail
File:FD Mazda RX-7.jpg
  artist: Aaron Lai
  license: CC BY 2.0 | https://creativecommons.org/licenses/by/2.0
  attribution required: true
  thumb: https

In [26]:
def clean(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()

rows = []
for p in r.json()["query"]["pages"].values():
    info = p["imageinfo"][0]
    meta = info["extmetadata"]
    rows.append({
        "title": p["title"],
        "artist": clean(meta.get("Artist", {}).get("value")),
        "license": meta.get("LicenseShortName", {}).get("value"),
        "license_url": meta.get("LicenseUrl", {}).get("value"),
        "thumb_url": info.get("thumburl"),
    })

rx7_images = pd.DataFrame(rows)
rx7_images["usable"] = rx7_images["license_url"].notna()
print(rx7_images[["title", "artist", "license", "usable"]])
print()
print(rx7_images["artist"].value_counts())

                                               title           artist  \
0               File:2000-2003 Mazda RX-7 engine.jpg           TTTNIS   
1               File:2017-06-10 Jason Clark RX-7.JPG    Lewis Collard   
2                             File:FD Mazda RX-7.jpg        Aaron Lai   
3                        File:FD Mazda RX-7 rear.jpg    Jacob Frey 4A   
4  File:Mazda RX-7 (FD) Washington DC Metro Area,...  OWS Photography   
5  File:Mazda RX-7 (FD) Washington DC Metro Area,...  OWS Photography   
6  File:Mazda RX-7 (FD) at the 2025 Shannons Adel...      Yu Chu Chin   
7  File:Mazda RX-7 (FD) at the Pavilion Hotel Kua...      Yu Chu Chin   
8  File:Mazda RX-7 (FD) at the Pavilion Hotel Kua...      Yu Chu Chin   
9  File:Mazda RX-7 of Mike Whiddett at the 2026 A...      Yu Chu Chin   

        license  usable  
0           CC0    True  
1   Attribution   False  
2     CC BY 2.0    True  
3     CC BY 2.0    True  
4     CC BY 4.0    True  
5     CC BY 4.0    True  
6     CC BY 4.

In [27]:
def search_images(query, limit=30, max_per_artist=3, skip_words=("engine", "interior", "dashboard")):
    params = {
        "action": "query", "format": "json",
        "generator": "search", "gsrsearch": query, "gsrnamespace": 6, "gsrlimit": limit,
        "prop": "imageinfo", "iiprop": "url|size|extmetadata", "iiurlwidth": 640,
    }
    r = requests.get(api, params=params, headers=headers, timeout=30)
    pages = r.json().get("query", {}).get("pages", {})
    rows = []
    for p in pages.values():
        if "imageinfo" not in p:
            continue
        info = p["imageinfo"][0]
        meta = info["extmetadata"]
        rows.append({
            "title": p["title"],
            "artist": clean(meta.get("Artist", {}).get("value")),
            "license": meta.get("LicenseShortName", {}).get("value"),
            "license_url": meta.get("LicenseUrl", {}).get("value"),
            "thumb_url": info.get("thumburl"),
        })
    time.sleep(0.5)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df[df["license_url"].notna()]
    df = df[~df["title"].str.lower().str.contains("|".join(skip_words))]
    return df.groupby("artist").head(max_per_artist).reset_index(drop=True)

s15 = search_images("Nissan Silvia S15")
print(len(s15), "usable images from", s15["artist"].nunique(), "photographers")
print(s15[["title", "artist", "license"]])

14 usable images from 9 photographers
                                                title                  artist  \
0                   File:1999 Nissan Silvia S15 1.jpg               Calreyn88   
1                   File:1999 Nissan Silvia S15 2.jpg               Calreyn88   
2                  File:1999 Nissan Silvia Spec R.jpg               Calreyn88   
3   File:Malaysia Female Drifter Minvern K Driftin...             KUAH YOU JI   
4   File:Moscow, Nissan Silvia (S15) spec-R, May 2...     Retired electrician   
5         File:Nissan Silvia S15 Spec R in Europe.png            Razorback 69   
6   File:Nissan Silvia S15 Spec S, front left, 07-...            Ethan Llamas   
7   File:Nissan Silvia S15 Spec S, rear left, 07-2...            Ethan Llamas   
8               File:Nissan Silvia S15 Spec S (1).jpg            Ethan Llamas   
9                File:Nissan Silvia S15 Teknofest.jpg                 Kuzel06   
10  File:Nissan Silvia S15 at the 2026 Adelaide Mo...             Yu Ch

In [28]:
from IPython.display import HTML

cells = "".join(
    f'<div style="display:inline-block;width:250px;margin:6px;vertical-align:top;font-size:12px">'
    f'<img src="{row["thumb_url"]}" width="250"><br>{i}: {row["title"][5:60]}</div>'
    for i, row in s15.iterrows()
)
HTML(cells)

In [29]:
s15["car_id"] = "silvia_s15"
s15["review"] = "keep"
s15["reason"] = ""

s15.loc[[3, 9], "review"] = "drop"
s15.loc[[0, 1], "review"] = "unsure"
s15.loc[3, "reason"] = "drifting action shot"
s15.loc[9, "reason"] = "custom paint"
s15.loc[[0, 1], "reason"] = "pink, unsure if stock paint"

print(s15["review"].value_counts())
s15.to_csv("data/image_manifest.csv", index=False)

review
keep      10
unsure     2
drop       2
Name: count, dtype: int64


In [30]:
queries = {
    "rx7_fd":      ["Mazda RX-7 FD"],
    "supra_a80":   ["Toyota Supra A80"],
    "eclipse_2g":  ["Mitsubishi Eclipse 2G"],
    "skyline_gtr": ["Nissan Skyline GT-R R33", "Nissan Skyline GT-R R34"],
    "nissan_350z": ["Nissan 350Z"],
}

batch = []
for car_id, qs in queries.items():
    frames = [d for d in (search_images(q) for q in qs) if not d.empty]
    if not frames:
        print(car_id, "-> no usable images found")
        continue
    found = pd.concat(frames).drop_duplicates("title")
    found = found.groupby("artist").head(3).head(10).copy()
    found["car_id"] = car_id
    batch.append(found)
    print(car_id, len(found), "images,", found["artist"].nunique(), "photographers")

batch = pd.concat(batch, ignore_index=True)
print()
print(batch[["car_id", "title", "license"]].to_string())

rx7_fd 10 images, 6 photographers
supra_a80 10 images, 7 photographers
eclipse_2g 1 images, 1 photographers
skyline_gtr 10 images, 5 photographers
nissan_350z 9 images, 4 photographers

         car_id                                                                                   title       license
0        rx7_fd                                                                File:1992 Mazda RX-7.jpg  CC BY-SA 4.0
1        rx7_fd                                                             File:1992 Mazda RX-7 RZ.jpg  CC BY-SA 4.0
2        rx7_fd                                                   File:1994 Mazda RX-7 FD White (1).jpg  CC BY-SA 4.0
3        rx7_fd                                                   File:1994 Mazda RX-7 FD White (2).jpg  CC BY-SA 4.0
4        rx7_fd                                                   File:1994 Mazda RX-7 FD White (3).jpg  CC BY-SA 4.0
5        rx7_fd                                                             File:1997 Mazda RX-7 RZ.jpg  C

In [31]:
tests = [
    "Mitsubishi Eclipse GSX",
    "Mitsubishi Eclipse 1995",
    "Mitsubishi Eclipse GS-T",
    "Mitsubishi Eclipse 1997",
    "Nissan Skyline GT-R R34",
]
for q in tests:
    d = search_images(q)
    n_artists = d["artist"].nunique() if not d.empty else 0
    print(q, "->", len(d), "images,", n_artists, "photographers")
    if not d.empty:
        print(d[["title"]].head(6).to_string(header=False))
    print()

Mitsubishi Eclipse GSX -> 7 images, 4 photographers
0                                                  File:'96 Mitsubishi Eclipse GSX.jpg
1  File:1990 Mitsubishi Eclipse GSX Turbo in Maranello Red, front left, 05-23-2026.jpg
2   File:1990 Mitsubishi Eclipse GSX Turbo in Maranello Red, rear left, 05-23-2026.jpg
3                                    File:1993 Mitsubishi Eclipse GSX (4815541158).jpg
4                                    File:1993 Mitsubishi Eclipse GSX (4815542974).jpg
5                          File:1993 Mitsubishi Eclipse GSX (4815542974) (cropped).jpg

Mitsubishi Eclipse 1995 -> 13 images, 10 photographers
0                                            File:Corsa Ferrari.jpg
1                                    File:Eclipse GT 08-14-2017.jpg
2                   File:Mitsubishi EclipseIII 02 (31657002390).jpg
3  File:Mitsubishi Eclipse (D53A) Washington DC Metro Area, USA.jpg
4                         File:Mitsubishi Eclipse Międzyzdroje1.JPG
5                         File

In [33]:
def members(cat, kind, limit=50):
    p = {"action": "query", "format": "json", "list": "categorymembers",
         "cmtitle": cat, "cmtype": kind, "cmlimit": limit}
    r = requests.get(api, params=p, headers=headers, timeout=30)
    return [m["title"] for m in r.json()["query"]["categorymembers"]]

for cat in ["Category:Mitsubishi Eclipse", "Category:Mitsubishi Eclipse (PJ platform)"]:
    print(cat)
    print("  subcategories:", members(cat, "subcat"))
    print("  files:", len(members(cat, "file")), "(counted up to 50)")
    print()

Category:Mitsubishi Eclipse
  subcategories: ['Category:Mitsubishi Eclipse (D platform)', 'Category:Mitsubishi Eclipse (PJ platform)', 'Category:Mitsubishi Eclipse (ST-22 platform)', 'Category:Mitsubishi Eclipse (PS platform)']
  files: 1 (counted up to 50)

Category:Mitsubishi Eclipse (PJ platform)
  subcategories: []
  files: 50 (counted up to 50)



In [34]:
for cat in ["Category:Mitsubishi Eclipse (PJ platform)", "Category:Mitsubishi Eclipse (D platform)"]:
    titles = members(cat, "file", limit=30)
    print(cat, "->", len(titles), "files shown")
    for t in titles:
        print("  ", t)
    print()

Category:Mitsubishi Eclipse (PJ platform) -> 30 files shown
   File:'96 Mitsubishi Eclipse GSX.jpg
   File:'97-'99 Mitsubishi Eclipse Liftback.JPG
   File:'97-'99 Mitsubishi Eclipse Spyder (Orange Julep '07).jpg
   File:'97-'99 Mitsubishi Eclipse.JPG
   File:'99 Mitsubishi Eclipse.JPG
   File:1996 Mitsubishi Eclipse GS (Forssa, Finland).jpg
   File:1996 Mitsubishi.JPG
   File:1997 Mitsubishi Eclipse 2.0 01.jpg
   File:1997 Mitsubishi Eclipse 2.0 02.jpg
   File:1997-1999 Mitsubishi Eclipse.JPG
   File:1997-99 Mitsubishi Eclipse.JPG
   File:1999 Minden Silver Pearl Mitsubishi Eclipse RS.jpg
   File:1999 Mitsubishi Eclipse GSX.jpg
   File:2gbEclipse.jpg
   File:2GMitsubishiEclipseGSX.jpg
   File:2nd-Mitsubishi-Eclipse.jpg
   File:97-99 Mitsubishi Eclipse convertible.jpg
   File:97-99 Mitsubishi Eclipse.jpg
   File:A-tuned-Mitsubishi-Eclipse-(2G).jpg
   File:Aug06-vol4 208 (223828218).jpg
   File:BoiseTempleRd.jpg
   File:Cars on display at the Car and Bike show (8964139).jpg
   File:Eclip

In [35]:
def category_images(cat, limit=50, max_per_artist=3,
                    skip_words=("engine", "interior", "dashboard", "intercooler")):
    params = {
        "action": "query", "format": "json",
        "generator": "categorymembers", "gcmtitle": cat, "gcmtype": "file", "gcmlimit": limit,
        "prop": "imageinfo", "iiprop": "url|size|extmetadata", "iiurlwidth": 640,
    }
    r = requests.get(api, params=params, headers=headers, timeout=30)
    pages = r.json().get("query", {}).get("pages", {})
    rows = []
    for p in pages.values():
        if "imageinfo" not in p:
            continue
        info = p["imageinfo"][0]
        meta = info["extmetadata"]
        rows.append({
            "title": p["title"],
            "artist": clean(meta.get("Artist", {}).get("value")),
            "license": meta.get("LicenseShortName", {}).get("value"),
            "license_url": meta.get("LicenseUrl", {}).get("value"),
            "thumb_url": info.get("thumburl"),
        })
    time.sleep(0.5)
    df = pd.DataFrame(rows)
    df = df[df["license_url"].notna()]
    df = df[~df["title"].str.lower().str.contains("|".join(skip_words))]
    return df.groupby("artist").head(max_per_artist).reset_index(drop=True)

eclipse2g = category_images("Category:Mitsubishi Eclipse (PJ platform)")
print(len(eclipse2g), "images from", eclipse2g["artist"].nunique(), "photographers")
print(eclipse2g[["title", "artist", "license"]].to_string())

28 images from 22 photographers
                                                                          title                                              artist       license
0                                           File:'96 Mitsubishi Eclipse GSX.jpg                                        The Dude 421  CC BY-SA 3.0
1                         File:1996 Mitsubishi Eclipse GS (Forssa, Finland).jpg                         Dogs.barking.duster.rolling  CC BY-SA 4.0
2                                                      File:1996 Mitsubishi.JPG                                      Dean Patterson  CC BY-SA 4.0
3                                       File:1997 Mitsubishi Eclipse 2.0 01.jpg                                 Rutger van der Maar     CC BY 2.0
4                                       File:1997 Mitsubishi Eclipse 2.0 02.jpg                                 Rutger van der Maar     CC BY 2.0
5                                          File:1999 Mitsubishi Eclipse GSX.jpg             

In [36]:
three = batch[batch["car_id"].isin(["rx7_fd", "supra_a80", "nissan_350z"])]
three = three[~three["title"].str.contains("Osaka Auto Messe|Z32", case=False)]

e = eclipse2g.groupby("artist").head(1).head(10).copy()
e["car_id"] = "eclipse_2g"

sk = []
for q in ["Nissan Skyline GT-R R33", "Nissan Skyline GT-R R34"]:
    d = search_images(q).head(5).copy()
    d["car_id"] = "skyline_gtr"
    sk.append(d)
sk = pd.concat(sk).drop_duplicates("title")

candidates = pd.concat([three, e, sk], ignore_index=True)
print(candidates.groupby("car_id").size())
print(len(candidates), "candidates total")

car_id
eclipse_2g     10
nissan_350z     8
rx7_fd         10
skyline_gtr    10
supra_a80       9
dtype: int64
47 candidates total


In [37]:
from IPython.display import HTML

def show(car_id):
    d = candidates[candidates["car_id"] == car_id]
    cells = "".join(
        f'<div style="display:inline-block;width:250px;margin:6px;vertical-align:top;font-size:12px">'
        f'<img src="{row["thumb_url"]}" width="250"><br>{i}: {row["title"][5:60]}</div>'
        for i, row in d.iterrows()
    )
    return HTML(cells)

show("rx7_fd")

In [ ]:
#Updated  Review Table
if "review" not in candidates.columns:
    candidates["review"] = ""
    candidates["reason"] = ""

candidates.loc[candidates["car_id"] == "rx7_fd", "review"] = "keep"

manifest = pd.read_csv("data/image_manifest.csv")
reviewed = candidates[candidates["review"] != ""]
manifest = pd.concat([manifest, reviewed], ignore_index=True)
manifest = manifest.drop_duplicates(["car_id", "title"], keep="last")
manifest.to_csv("data/image_manifest.csv", index=False)

print(manifest.groupby(["car_id", "review"]).size())

car_id      review
rx7_fd      keep      10
silvia_s15  drop       2
            keep      10
            unsure     2
dtype: int64


In [39]:
show("supra_a80")

In [40]:
supra = candidates["car_id"] == "supra_a80"
candidates.loc[supra, "review"] = "keep"
candidates.loc[10, "reason"] = "custom hood, still recognizable"

manifest = pd.read_csv("data/image_manifest.csv")
reviewed = candidates[candidates["review"] != ""]
manifest = pd.concat([manifest, reviewed], ignore_index=True)
manifest = manifest.drop_duplicates(["car_id", "title"], keep="last")
manifest.to_csv("data/image_manifest.csv", index=False)

print(manifest.groupby(["car_id", "review"]).size())

car_id      review
rx7_fd      keep      10
silvia_s15  drop       2
            keep      10
            unsure     2
supra_a80   keep       9
dtype: int64


In [41]:
show("nissan_350z")

In [42]:
n350 = candidates["car_id"] == "nissan_350z"
candidates.loc[n350, "review"] = "keep"
candidates.loc[24, ["review", "reason"]] = ["drop", "night-time photo, poor quality"]
candidates.loc[25, ["review", "reason"]] = ["unsure", "D.K. pop-culture car; good for vibe, unsure for stock ID"]

manifest = pd.read_csv("data/image_manifest.csv")
reviewed = candidates[candidates["review"] != ""]
manifest = pd.concat([manifest, reviewed], ignore_index=True)
manifest = manifest.drop_duplicates(["car_id", "title"], keep="last")
manifest.to_csv("data/image_manifest.csv", index=False)

print(manifest.groupby(["car_id", "review"]).size())

car_id       review
nissan_350z  drop       1
             keep       6
             unsure     1
rx7_fd       keep      10
silvia_s15   drop       2
             keep      10
             unsure     2
supra_a80    keep       9
dtype: int64


In [43]:
extra = eclipse2g[eclipse2g["title"].str.contains("Fast and the Furious|Fast and Furious", case=False)].copy()
extra["car_id"] = "eclipse_2g"
extra["review"] = ""
extra["reason"] = ""

candidates = pd.concat([candidates, extra], ignore_index=True).drop_duplicates(["car_id", "title"])
show("eclipse_2g")

In [44]:
ec = candidates["car_id"] == "eclipse_2g"
candidates.loc[ec, "review"] = "keep"

candidates.loc[31, ["review", "reason"]] = ["drop", "engine photo"]
candidates.loc[32, ["review", "reason"]] = ["drop", "heavy modification, scissor doors"]
candidates.loc[34, ["review", "reason"]] = ["drop", "traffic light photo, not the car"]
for i in [36, 47, 48, 49]:
    candidates.loc[i, ["review", "reason"]] = ["unsure", "custom paint and heavy modification; same car(s) across photos"]

manifest = pd.read_csv("data/image_manifest.csv")
reviewed = candidates[candidates["review"] != ""]
manifest = pd.concat([manifest, reviewed], ignore_index=True)
manifest = manifest.drop_duplicates(["car_id", "title"], keep="last")
manifest.to_csv("data/image_manifest.csv", index=False)

print(manifest.groupby(["car_id", "review"]).size())

car_id       review
eclipse_2g   drop       3
             keep       6
             unsure     4
nissan_350z  drop       1
             keep       6
             unsure     1
rx7_fd       keep      10
silvia_s15   drop       2
             keep      10
             unsure     2
supra_a80    keep       9
dtype: int64


In [45]:
show("skyline_gtr")

In [46]:
sk_mask = candidates["car_id"] == "skyline_gtr"
candidates.loc[sk_mask, "review"] = "keep"
candidates.loc[37, ["review", "reason"]] = ["drop", "edited black-and-white photo"]
candidates.loc[46, ["review", "reason"]] = ["drop", "diecast model, not a real car"]

manifest = pd.read_csv("data/image_manifest.csv")
reviewed = candidates[candidates["review"] != ""]
manifest = pd.concat([manifest, reviewed], ignore_index=True)
manifest = manifest.drop_duplicates(["car_id", "title"], keep="last")
manifest.to_csv("data/image_manifest.csv", index=False)

print(manifest.groupby(["car_id", "review"]).size())
print()
print(manifest["review"].value_counts())

car_id       review
eclipse_2g   drop       3
             keep       6
             unsure     4
nissan_350z  drop       1
             keep       6
             unsure     1
rx7_fd       keep      10
silvia_s15   drop       2
             keep      10
             unsure     2
skyline_gtr  drop       2
             keep       8
supra_a80    keep       9
dtype: int64

review
keep      49
drop       8
unsure     7
Name: count, dtype: int64
